# SP-8502: Métodos Cuanti-Cuali con IA Responsable
## Sesión 9 - Sprint 3: Regresión Lineal - Supuestos, Diagnóstico y Modelado

**Laboratorio Autoguiado:** Análisis de Captura de Peces en Pesquerías Artesanales

**Región:** Pacífico Central, Costa Rica (Datos Sintéticos)

**Nivel:** PhD - Interpretación detallada, supuestos verificados, diagnósticos exhaustivos

---

### Objetivo
Dominar:
1. Regresión lineal simple y múltiple
2. Verificación de supuestos (linealidad, normalidad, homocedasticidad, independencia)
3. Diagnósticos con visualizaciones y tests estadísticos
4. Multicolinealidad (VIF, correlaciones)
5. Transformaciones (log, sqrt, estandarización)
6. Detección de outliers e influyentes
7. Comparación de modelos


## INSTALACIÓN Y CONFIGURACIÓN

In [ ]:
# Instalar dependencias necesarias (ejecutar UNA SOLA VEZ)
# Nota técnica: el nombre para instalar un paquete con pip no siempre coincide
# con el nombre usado para importarlo en Python. Por ejemplo:
#   pip install scikit-learn  ->  import sklearn
import importlib
import subprocess
import sys

packages = {
    'numpy': 'numpy',
    'pandas': 'pandas',
    'matplotlib': 'matplotlib',
    'seaborn': 'seaborn',
    'scipy': 'scipy',
    'statsmodels': 'statsmodels',
    'scikit-learn': 'sklearn'
}

for pip_name, import_name in packages.items():
    print(f"Verificando {pip_name}...")
    try:
        importlib.import_module(import_name)
        print(f"  ✓ {pip_name} ya instalado")
    except ImportError:
        print(f"  Instalando {pip_name}...")
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pip_name, '-q'])
        print(f"  ✓ {pip_name} instalado")


In [ ]:
# Importar librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import shapiro, levene
from statsmodels.stats.outliers_influence import variance_inflation_factor as VIF
from statsmodels.graphics.gofplots import ProbPlot
import statsmodels.api as sm
from statsmodels.formula.api import ols
import warnings
warnings.filterwarnings('ignore')

# Configuración visual robusta para Google Colab
try:
    plt.style.use('seaborn-v0_8-darkgrid')
except OSError:
    plt.style.use('default')
sns.set_palette('husl')
%matplotlib inline

print("\n" + "="*80)
print("LABORATORIO AUTOGUIADO: REGRESIÓN LINEAL")
print("Sesión 9 - Sprint 3: SP-8502")
print("="*80)


## PARTE 1: GENERACIÓN DE DATASET SINTÉTICO REALISTA

In [ ]:
print("\n[PARTE 1] Generación del Dataset Sintético")
print("-" * 80)

np.random.seed(42)  # Para reproducibilidad
n_observations = 150

# Simulación realista: captura de peces artesanales en Pacífico Central
data = {
    'id_pescador': np.arange(1, n_observations + 1),
    
    # Variable dependiente (Y)
    'captura_kg': None,  # A generar
    
    # Variables independientes (X)
    'horas_faena': np.random.normal(loc=6.5, scale=1.2, size=n_observations),
    'tamanio_embarcacion_m': np.random.normal(loc=7.2, scale=2.1, size=n_observations),
    'experiencia_anios': np.random.uniform(1, 45, size=n_observations),
    'precio_hielo_colones': np.random.normal(loc=3500, scale=500, size=n_observations),
    'temperatura_agua_C': np.random.normal(loc=24.5, scale=1.8, size=n_observations),
}

# Construcción teórica de Y (Data Generating Process conocido)
# Y = β₀ + β₁*X₁ + β₂*X₂ + β₃*X₃ + ε
beta_0 = 15.0
beta_1 = 8.5
beta_2 = 3.2
beta_3 = 0.4

epsilon = np.random.normal(loc=0, scale=5.0, size=n_observations)

data['captura_kg'] = (
    beta_0 + 
    beta_1 * data['horas_faena'] + 
    beta_2 * data['tamanio_embarcacion_m'] + 
    beta_3 * data['experiencia_anios'] + 
    epsilon
)

# Restricción de dominio: captura > 0 (realismo)
data['captura_kg'] = np.maximum(data['captura_kg'], 5)

df = pd.DataFrame(data)

print(f"✓ Dataset creado: {df.shape[0]} observaciones, {df.shape[1]} variables")
print(f"\nEstadísticas Descriptivas:")
print(df[['captura_kg', 'horas_faena', 'tamanio_embarcacion_m', 'experiencia_anios', 'temperatura_agua_C']].describe().round(3))

print(f"\nPrimeras 5 filas:")
print(df.head())

## PARTE 2: EXPLORACIÓN Y CORRELACIONES

In [ ]:
print("\n[PARTE 2] Exploración Inicial - Matriz de Correlaciones")
print("-" * 80)

# Seleccionar variables numéricas
numeric_cols = ['captura_kg', 'horas_faena', 'tamanio_embarcacion_m', 
                'experiencia_anios', 'temperatura_agua_C']
corr_matrix = df[numeric_cols].corr()

print("\nMatriz de Correlaciones (Pearson):")
print(corr_matrix.round(3))

# Visualización
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='coolwarm', center=0,
            square=True, linewidths=1, cbar_kws={"shrink": 0.8}, ax=ax, vmin=-1, vmax=1)
ax.set_title('Matriz de Correlaciones - Dataset de Pesquerías Artesanales\n' + 
             'Captura vs. Factores Técnicos, Sociales y Ambientales', 
             fontsize=12, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

print("\n✓ Interpretación:")
print("  Todas las correlaciones con captura_kg son positivas.")
print("  Esto sugiere que: más horas, mayor embarcación, más experiencia → mayor captura")

## PARTE 3: REGRESIÓN LINEAL SIMPLE

In [ ]:
print("\n[PARTE 3] Regresión Lineal Simple: Horas de Faena → Captura")
print("-" * 80)

# Ajustar modelo simple
modelo_simple = ols('captura_kg ~ horas_faena', data=df).fit()

print("\n" + modelo_simple.summary().as_text())

# Visualización
fig, ax = plt.subplots(figsize=(11, 6))
ax.scatter(df['horas_faena'], df['captura_kg'], alpha=0.6, s=70, 
          color='steelblue', edgecolors='black', linewidth=0.5, label='Datos observados')

# Línea de predicción
x_range = np.linspace(df['horas_faena'].min(), df['horas_faena'].max(), 100)
y_pred = modelo_simple.predict(pd.DataFrame({'horas_faena': x_range}))
ax.plot(x_range, y_pred, 'r-', linewidth=2.5, label=f'Línea ajustada (R² = {modelo_simple.rsquared:.3f})')

ax.set_xlabel('Horas de Faena', fontsize=11, fontweight='bold')
ax.set_ylabel('Captura (kg)', fontsize=11, fontweight='bold')
ax.set_title('Regresión Lineal Simple: Horas de Faena vs Captura\n' +
             f'Ecuación: Captura = {modelo_simple.params.iloc[0]:.2f} + {modelo_simple.params.iloc[1]:.2f}*Horas',
             fontsize=12, fontweight='bold', pad=15)
ax.legend(fontsize=10, loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n✓ Interpretación de Coeficientes:")
print(f"  • Intercepto (β₀) = {modelo_simple.params[0]:.3f} kg")
print(f"    → Captura esperada cuando horas = 0 (valor teórico)")
print(f"  • Pendiente (β₁) = {modelo_simple.params.iloc[1]:.3f} kg/hora")
print(f"    → INTERPRETACIÓN: Por cada hora adicional de faena,")
print(f"      se espera un aumento de {modelo_simple.params.iloc[1]:.3f} kg de captura (ceteris paribus)")
print(f"  • R² = {modelo_simple.rsquared:.3f}")
print(f"    → El modelo explica {modelo_simple.rsquared*100:.1f}% de la variabilidad en captura")
print(f"  • p-value (F-test) = {modelo_simple.f_pvalue:.2e}")
print(f"    → El modelo es significativo (p < 0.05)")

## PARTE 4: REGRESIÓN LINEAL MÚLTIPLE

In [ ]:
print("\n[PARTE 4] Regresión Lineal Múltiple")
print("-" * 80)

# Ajustar modelo múltiple con 3 predictores
modelo_multiple = ols('captura_kg ~ horas_faena + tamanio_embarcacion_m + experiencia_anios', 
                       data=df).fit()

print("\nEspecificación del Modelo:")
print("captura_kg = β₀ + β₁*horas_faena + β₂*tamanio_embarcacion_m + β₃*experiencia_anios + ε")
print("\n" + modelo_multiple.summary().as_text())

print("\n✓ Interpretación de Coeficientes (manteniendo otros constantes - ceteris paribus):")
for var in modelo_multiple.params.index[1:]:
    coef = modelo_multiple.params[var]
    pval = modelo_multiple.pvalues[var]
    se = modelo_multiple.bse[var]
    t_stat = modelo_multiple.tvalues[var]
    sig = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else "ns"
    print(f"\n  • {var}:")
    print(f"    Coef = {coef:.4f} {sig} (SE = {se:.4f}, t = {t_stat:.3f}, p = {pval:.4f})")
    
print(f"\n✓ Ajuste Global del Modelo:")
print(f"  • R² = {modelo_multiple.rsquared:.4f}")
print(f"    → {modelo_multiple.rsquared*100:.2f}% de variabilidad en captura explicada")
print(f"  • R² Ajustado = {modelo_multiple.rsquared_adj:.4f}")
print(f"    → Ajustado por número de predictores (penaliza sobre-parametrización)")
print(f"  • F-statistic = {modelo_multiple.fvalue:.2f}, p-value = {modelo_multiple.f_pvalue:.2e}")
print(f"    → El modelo en conjunto es significativo")

print(f"\n✓ Comparación: Modelo Simple vs. Múltiple")
print(f"  R² Simple:   {modelo_simple.rsquared:.4f}")
print(f"  R² Múltiple: {modelo_multiple.rsquared:.4f}")
print(f"  Mejora: {(modelo_multiple.rsquared - modelo_simple.rsquared):.4f}")
print(f"  → Agregar predictores mejora significativamente el modelo")

## PARTE 5: VERIFICACIÓN DE SUPUESTOS (DIAGNÓSTICOS EXHAUSTIVOS)

### 5.1: SUPUESTO DE LINEALIDAD

In [ ]:
print("\n[5.1] Supuesto de Linealidad")
print("-" * 80)
print("Método: Gráfico de Residuales vs. Valores Ajustados")
print("Hipótesis nula: La relación entre Y y X es lineal")
print("Evidencia de violación: Residuales muestran patrón curvo")

# Calcular residuales y valores ajustados
residuals = modelo_multiple.resid
fitted_values = modelo_multiple.fittedvalues

fig, ax = plt.subplots(figsize=(11, 6))
ax.scatter(fitted_values, residuals, alpha=0.6, s=70, 
          color='steelblue', edgecolors='black', linewidth=0.5)
ax.axhline(y=0, color='r', linestyle='--', linewidth=2.5, label='Línea Y=0')

# Suavizado LOWESS para detectar patrones no lineales
from scipy.signal import savgol_filter
sorted_idx = np.argsort(fitted_values)
window_length = min(51, len(fitted_values) // 2 * 2 + 1)
try:
    y_smooth = savgol_filter(residuals.iloc[sorted_idx].values, window_length, 3)
    ax.plot(fitted_values.iloc[sorted_idx], y_smooth, 'g-', linewidth=2.5, 
           label='Tendencia (Savitzky-Golay)', alpha=0.8)
except:
    pass

ax.set_xlabel('Valores Ajustados (ŷ)', fontsize=11, fontweight='bold')
ax.set_ylabel('Residuales (e)', fontsize=11, fontweight='bold')
ax.set_title('Diagnóstico 1: Residuales vs. Valores Ajustados\n' +
             'Verifica: Linealidad (patrón aleatorio alrededor de y=0)',
             fontsize=12, fontweight='bold', pad=15)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n✓ Criterios de Evaluación:")
print("  ✓ LINEALIDAD OK: Residuales dispersan aleatoriamente sin patrón")
print("  ✗ LINEALIDAD VIOLADA: Residuales muestran patrón curvo/cuadrático")
print("\nAcción si violado: Considerar términos polinomiales, transformar Y, o usar splines")

### 5.2: SUPUESTO DE NORMALIDAD DE RESIDUALES

In [ ]:
print("\n[5.2] Supuesto de Normalidad de Residuales")
print("-" * 80)
print("Hipótesis nula: ε ~ Normal(0, σ²)")
print("Importancia: Crítica para intervalos de confianza e inferencia en muestras pequeñas")

fig, axes = plt.subplots(2, 2, figsize=(13, 11))

# Panel 1: Histograma
axes[0, 0].hist(residuals, bins=20, color='steelblue', edgecolor='black', alpha=0.7, density=True)
mu, sigma = residuals.mean(), residuals.std()
x = np.linspace(residuals.min(), residuals.max(), 100)
axes[0, 0].plot(x, stats.norm.pdf(x, mu, sigma), 'r-', linewidth=2.5, label='Densidad Normal')
axes[0, 0].set_xlabel('Residuales', fontsize=10, fontweight='bold')
axes[0, 0].set_ylabel('Densidad', fontsize=10, fontweight='bold')
axes[0, 0].set_title('(A) Histograma de Residuales', fontsize=11, fontweight='bold')
axes[0, 0].legend(fontsize=9)
axes[0, 0].grid(True, alpha=0.3)

# Panel 2: Q-Q Plot
pp = ProbPlot(residuals)
pp.qqplot(ax=axes[0, 1], line='45', alpha=0.6, markersize=7)
axes[0, 1].set_title('(B) Q-Q Plot: Residuales vs. Normal Teórica', fontsize=11, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)

# Panel 3: Test Shapiro-Wilk
stat_sw, pval_sw = shapiro(residuals)
axes[1, 0].text(0.5, 0.7, 'Test Shapiro-Wilk', ha='center', fontsize=12, fontweight='bold',
                transform=axes[1, 0].transAxes)
axes[1, 0].text(0.5, 0.5, f'Estadístico W: {stat_sw:.4f}\np-value: {pval_sw:.4f}',
                ha='center', fontsize=11, transform=axes[1, 0].transAxes,
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.7))
axes[1, 0].text(0.5, 0.15, 'H₀: Residuales ~ Normal\n' + 
                ('✗ RECHAZO H₀ (no normales, p < 0.05)' if pval_sw < 0.05 else '✓ NO RECHAZO H₀ (normales, p ≥ 0.05)'),
                ha='center', fontsize=10, transform=axes[1, 0].transAxes,
                style='italic', fontweight='bold')
axes[1, 0].axis('off')

# Panel 4: ECDF
from scipy.stats import norm
sorted_residuals = np.sort(residuals)
ecdf = np.arange(1, len(sorted_residuals) + 1) / len(sorted_residuals)
normal_ecdf = norm.cdf(sorted_residuals, mu, sigma)
axes[1, 1].plot(sorted_residuals, ecdf, 'b-', linewidth=2.5, label='ECDF empírica', marker='o', markersize=3, alpha=0.7)
axes[1, 1].plot(sorted_residuals, normal_ecdf, 'r--', linewidth=2.5, label='CDF Normal teórica')
axes[1, 1].set_xlabel('Residuales', fontsize=10, fontweight='bold')
axes[1, 1].set_ylabel('Probabilidad Acumulada', fontsize=10, fontweight='bold')
axes[1, 1].set_title('(D) Función de Distribución Acumulada (ECDF)', fontsize=11, fontweight='bold')
axes[1, 1].legend(fontsize=9, loc='upper left')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n✓ Test Shapiro-Wilk:")
print(f"  Estadístico W = {stat_sw:.4f}")
print(f"  p-value = {pval_sw:.4f}")
if pval_sw < 0.05:
    print(f"  ✗ RECHAZO H₀: Residuales NO son normales (p < 0.05)")
    print(f"    Alternativas a explorar:")
    print(f"      1. Transformar Y (log, sqrt, Box-Cox)")
    print(f"      2. Usar GLM/Generalized Linear Models")
    print(f"      3. Robust regression (M-estimators)")
    print(f"      4. Bootstrapping para intervalos de confianza")
else:
    print(f"  ✓ NO RECHAZO H₀: Residuales son normales (p ≥ 0.05)")

### 5.3: SUPUESTO DE HOMOCEDASTICIDAD (VARIANZA CONSTANTE)

In [ ]:
print("\n[5.3] Supuesto de Homocedasticidad (Varianza Constante)")
print("-" * 80)
print("Hipótesis nula: Var(ε) = σ² (constante para todos los valores de X)")
print("Violación (heterocedasticidad): Var(ε|X) cambia con X")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: Scale-Location Plot
standardized_residuals = residuals / residuals.std()
sqrt_abs_std_resid = np.sqrt(np.abs(standardized_residuals))

axes[0].scatter(fitted_values, sqrt_abs_std_resid, alpha=0.6, s=70, 
               color='steelblue', edgecolors='black', linewidth=0.5)
axes[0].axhline(y=np.sqrt(2), color='r', linestyle='--', linewidth=2, label='Línea referencia')
axes[0].set_xlabel('Valores Ajustados (ŷ)', fontsize=11, fontweight='bold')
axes[0].set_ylabel('√|Residuales Estandarizados|', fontsize=11, fontweight='bold')
axes[0].set_title('(A) Scale-Location Plot\nVerifica varianza constante',
                 fontsize=11, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Panel 2: Test de Levene
median_fitted = fitted_values.median()
group1 = residuals[fitted_values < median_fitted]
group2 = residuals[fitted_values >= median_fitted]
stat_levene, pval_levene = levene(group1, group2)

axes[1].text(0.5, 0.7, 'Test de Levene', ha='center', fontsize=12, fontweight='bold',
            transform=axes[1].transAxes)
axes[1].text(0.5, 0.5, f'Estadístico: {stat_levene:.4f}\np-value: {pval_levene:.4f}',
            ha='center', fontsize=11, transform=axes[1].transAxes,
            bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.7))
axes[1].text(0.5, 0.15, 'H₀: Varianza igual en todos los niveles\n' +
            ('✗ RECHAZO H₀ (varianza no constante)' if pval_levene < 0.05 else '✓ NO RECHAZO H₀ (varianza constante)'),
            ha='center', fontsize=10, transform=axes[1].transAxes, style='italic', fontweight='bold')
axes[1].axis('off')

plt.tight_layout()
plt.show()

print(f"\n✓ Test de Levene:")
print(f"  Estadístico = {stat_levene:.4f}")
print(f"  p-value = {pval_levene:.4f}")
if pval_levene < 0.05:
    print(f"  ✗ RECHAZO H₀: Varianza NO es constante (heterocedasticidad)")
    print(f"    Alternativas:")
    print(f"      1. Transformar Y (log, sqrt)")
    print(f"      2. Weighted Least Squares (WLS)")
    print(f"      3. Errores Estándar Robustos (HC1, HC2, HC3)")
else:
    print(f"  ✓ NO RECHAZO H₀: Varianza es constante (homocedasticidad OK)")

### 5.4: SUPUESTO DE INDEPENDENCIA DE RESIDUALES

In [ ]:
print("\n[5.4] Supuesto de Independencia de Residuales")
print("-" * 80)
print("Hipótesis nula: Cov(εᵢ, εⱼ) = 0 para i ≠ j")
print("Nota: En datos transversales (cross-sectional), asumir independencia.")
print("      En series de tiempo, usar Durbin-Watson o Box-Ljung test.")

# Durbin-Watson
from statsmodels.stats.stattools import durbin_watson
dw = durbin_watson(residuals)

print(f"\nDurbin-Watson: {dw:.4f}")
print(f"Interpretación (rango: [0, 4]):")
print(f"  DW ≈ 2:   Residuales independientes (OK)")
print(f"  DW < 2:   Autocorrelación positiva (residuales correlacionados)")
print(f"  DW > 2:   Autocorrelación negativa")

# Gráfico: Residuales en orden
fig, ax = plt.subplots(figsize=(13, 6))
ax.plot(range(len(residuals)), residuals.values, marker='o', linestyle='-', 
       alpha=0.6, color='steelblue', markersize=5, label='Residuales')
ax.axhline(y=0, color='r', linestyle='--', linewidth=2)
ax.fill_between(range(len(residuals)), -2*residuals.std(), 2*residuals.std(), 
                alpha=0.2, color='gray', label='±2 SD')
ax.set_xlabel('Orden de Observación (ID Pescador)', fontsize=11, fontweight='bold')
ax.set_ylabel('Residual', fontsize=11, fontweight='bold')
ax.set_title('Gráfico de Secuencia: Residuales en Orden\n' +
            'Verifica: Patrón aleatorio (sin tendencias, clusters, ciclos)',
            fontsize=12, fontweight='bold', pad=15)
ax.legend(fontsize=10, loc='upper right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n✓ Evaluación Visual:")
print(f"  Si puntos dispersan aleatoriamente → independencia OK")
print(f"  Si hay patrón cíclico/tendencia → autocorrelación presente")

## PARTE 6: ANÁLISIS DE MULTICOLINEALIDAD

In [ ]:
print("\n[PARTE 6] Análisis de Multicolinealidad")
print("-" * 80)

# Preparar datos para VIF
X_numeric = df[['horas_faena', 'tamanio_embarcacion_m', 'experiencia_anios']]
X_with_const = sm.add_constant(X_numeric)

# Calcular VIF
vif_data = pd.DataFrame()
vif_data["Variable"] = X_numeric.columns
vif_data["VIF"] = [VIF(X_with_const.values, i) for i in range(1, X_with_const.shape[1])]
vif_data = vif_data.sort_values('VIF', ascending=False).reset_index(drop=True)

print("\nVariance Inflation Factor (VIF):")
print(vif_data.to_string(index=False))

print("\nGrillas de Interpretación VIF:")
print("  VIF < 5:     ✓ Multicolinealidad baja → OK, sin problemas")
print("  VIF 5-10:    ⚠ Multicolinealidad moderada → revisar")
print("  VIF > 10:    ✗ Multicolinealidad severa → eliminar/combinar predictores")

# Matriz de correlaciones entre predictores
fig, ax = plt.subplots(figsize=(10, 8))
corr_predictors = X_numeric.corr()
sns.heatmap(corr_predictors, annot=True, fmt='.3f', cmap='coolwarm', center=0,
            square=True, linewidths=2, cbar_kws={"shrink": 0.8}, ax=ax,
            vmin=-1, vmax=1, annot_kws={"fontsize": 11})
ax.set_title('Matriz de Correlaciones: Predictores\n' +
            'Valores cercanos a ±1 sugieren multicolinealidad',
            fontsize=12, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

print("\n✓ Conclusión sobre Multicolinealidad:")
max_vif = vif_data['VIF'].max()
if max_vif < 5:
    print(f"  ✓ Todos los VIF < 5 → Multicolinealidad NO es problema")
elif max_vif < 10:
    print(f"  ⚠ Algunos VIF 5-10 → Multicolinealidad moderada, revisar especificación")
else:
    print(f"  ✗ VIF > 10 detectado → Multicolinealidad severa, considerar:")
    print(f"    - Eliminar una variable redundante")
    print(f"    - PCA (Principal Component Regression)")
    print(f"    - Ridge/Lasso regression")

## PARTE 7: TRANSFORMACIONES DE VARIABLES

In [ ]:
print("\n[PARTE 7] Transformaciones de Variables")
print("-" * 80)

print("\n[7.1] Transformación Log de la Variable Dependiente")
print("Justificación: Mejorar normalidad y homocedasticidad")
print("              Interpretación más natural (elasticidades)")

# Crear variable transformada
df['log_captura'] = np.log(df['captura_kg'])

# Ajustar modelo con log(Y)
modelo_log = ols('log_captura ~ horas_faena + tamanio_embarcacion_m + experiencia_anios',
                  data=df).fit()

# Test de normalidad para ambos modelos
residuals_original = modelo_multiple.resid
residuals_log = modelo_log.resid
stat_sw_original, pval_sw_original = shapiro(residuals_original)
stat_sw_log, pval_sw_log = shapiro(residuals_log)

print(f"\nComparación de Ajuste:")
print(f"  Modelo Original (Y):      R² = {modelo_multiple.rsquared:.4f}")
print(f"  Modelo con log(Y):        R² = {modelo_log.rsquared:.4f}")
print(f"\nTest Shapiro-Wilk (Normalidad de Residuales):")
print(f"  Original: W = {stat_sw_original:.4f}, p = {pval_sw_original:.4f}")
print(f"  log(Y):   W = {stat_sw_log:.4f}, p = {pval_sw_log:.4f}")
if pval_sw_log > pval_sw_original:
    print(f"  ✓ Log transformation mejora normalidad (p aumenta)")
else:
    print(f"  ✗ Log transformation NO mejora normalidad")

# Visualizar distribución de residuales
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Original
axes[0].hist(residuals_original, bins=20, color='steelblue', alpha=0.7, 
            edgecolor='black', density=True)
mu1, sigma1 = residuals_original.mean(), residuals_original.std()
x1 = np.linspace(residuals_original.min(), residuals_original.max(), 100)
axes[0].plot(x1, stats.norm.pdf(x1, mu1, sigma1), 'r-', linewidth=2.5)
axes[0].set_title(f'(A) Modelo Original\nShapiro-Wilk p = {pval_sw_original:.4f}', 
                 fontsize=11, fontweight='bold')
axes[0].set_xlabel('Residuales', fontsize=10, fontweight='bold')
axes[0].set_ylabel('Densidad', fontsize=10, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Log transform
axes[1].hist(residuals_log, bins=20, color='lightgreen', alpha=0.7,
            edgecolor='black', density=True)
mu2, sigma2 = residuals_log.mean(), residuals_log.std()
x2 = np.linspace(residuals_log.min(), residuals_log.max(), 100)
axes[1].plot(x2, stats.norm.pdf(x2, mu2, sigma2), 'r-', linewidth=2.5)
axes[1].set_title(f'(B) Modelo con log(Captura)\nShapiro-Wilk p = {pval_sw_log:.4f}', 
                 fontsize=11, fontweight='bold')
axes[1].set_xlabel('Residuales', fontsize=10, fontweight='bold')
axes[1].set_ylabel('Densidad', fontsize=10, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nInterpretación de Coeficientes en Modelo log(Y):")
for var in modelo_log.params.index[1:]:
    coef = modelo_log.params[var]
    pval = modelo_log.pvalues[var]
    print(f"  {var}: {coef:.4f}")
    print(f"    → Elasticidad: aumento de 1% en {var} → cambio de {coef:.3f}% en Captura")

### 7.2: Estandarización de Predictores

In [ ]:
print("\n[7.2] Estandarización de Predictores")
print("Justificación:")
print("  • Facilita comparación de importancia relativa de X")
print("  • Mejora estabilidad numérica")
print("  • Como solo se estandarizan las X, los coeficientes quedan en kg por 1 desviación estándar de X")

# Estandarizar predictores
# ddof=0 usa la desviación estándar poblacional; ddof=1 también sería válido.
X_scaled = (X_numeric - X_numeric.mean()) / X_numeric.std(ddof=0)
X_scaled.columns = [col + '_scaled' for col in X_scaled.columns]
df_scaled = pd.concat([df[['captura_kg']], X_scaled], axis=1)

# Ajustar modelo con variables escaladas
modelo_scaled = ols('captura_kg ~ horas_faena_scaled + tamanio_embarcacion_m_scaled + experiencia_anios_scaled',
                     data=df_scaled).fit()

print("\nCoeficientes con predictores estandarizados:")
for var in modelo_scaled.params.index[1:]:
    coef = modelo_scaled.params[var]
    pval = modelo_scaled.pvalues[var]
    print(f"  {var}: {coef:.4f} kg por 1 SD de {var.replace('_scaled', '')} (p = {pval:.4f})")

print("\nInterpretación:")
print("  Aumento de 1 desviación estándar en X → cambio de β en kg de captura.")
print("  Esto permite comparar qué predictor tiene mayor peso relativo sobre Y.")

# Comparar magnitudes de coeficientes estandarizados
std_coefs = pd.DataFrame({
    'Variable': modelo_scaled.params.index[1:],
    'Coef_Scaled': modelo_scaled.params.values[1:],
    'Abs_Coef': np.abs(modelo_scaled.params.values[1:])
}).sort_values('Abs_Coef', ascending=False)

print("\nRanking de Importancia Relativa:")
for idx, row in std_coefs.iterrows():
    var_short = row['Variable'].replace('_scaled', '')
    print(f"  {var_short}: {row['Coef_Scaled']:.4f} kg por 1 SD")


## PARTE 8: OUTLIERS E INFLUENCIA

In [ ]:
print("\n[PARTE 8] Detección de Outliers y Observaciones Influyentes")
print("-" * 80)

# Calcular medidas de influencia
# Corrección clave:
# En statsmodels el atributo correcto es `hat_matrix_diag`, no `hat_diag`.
# Además, se usan residuales studentizados internos, más apropiados para diagnóstico de outliers.
influence = modelo_multiple.get_influence()
cooks_d = influence.cooks_distance[0]
leverage = influence.hat_matrix_diag
dfbetas = influence.dfbetas
standardized_residuals = influence.resid_studentized_internal

# Número de parámetros estimados, incluyendo intercepto.
p_parametros = int(modelo_multiple.df_model + 1)

# Visualizar cuatro diagnósticos
fig, axes = plt.subplots(2, 2, figsize=(14, 11))

# Panel 1: Cook's Distance
axes[0, 0].stem(range(len(cooks_d)), cooks_d, markerfmt=',', basefmt=' ')
axes[0, 0].axhline(y=4/len(df), color='r', linestyle='--', linewidth=2, label=f'Umbral: 4/n = {4/len(df):.3f}')
axes[0, 0].set_xlabel('Observación ID', fontsize=10, fontweight='bold')
axes[0, 0].set_ylabel("Cook's Distance", fontsize=10, fontweight='bold')
axes[0, 0].set_title("(A) Cook's Distance: Influencia de Observaciones", fontsize=11, fontweight='bold')
axes[0, 0].legend(fontsize=9)
axes[0, 0].grid(True, alpha=0.3)

# Panel 2: Residuales Studentizados
axes[0, 1].scatter(range(len(standardized_residuals)), standardized_residuals, alpha=0.6, s=60)
axes[0, 1].axhline(y=2, color='r', linestyle='--', linewidth=1.5, label='±2')
axes[0, 1].axhline(y=-2, color='r', linestyle='--', linewidth=1.5)
axes[0, 1].axhline(y=3, color='darkred', linestyle='--', linewidth=1.5, alpha=0.5, label='±3 severo')
axes[0, 1].axhline(y=-3, color='darkred', linestyle='--', linewidth=1.5, alpha=0.5)
axes[0, 1].set_xlabel('Observación ID', fontsize=10, fontweight='bold')
axes[0, 1].set_ylabel('Residual studentizado interno', fontsize=10, fontweight='bold')
axes[0, 1].set_title('(B) Residuales Studentizados (Outliers si |r| > 2)', fontsize=11, fontweight='bold')
axes[0, 1].legend(fontsize=9)
axes[0, 1].grid(True, alpha=0.3)

# Panel 3: Leverage
axes[1, 0].scatter(range(len(leverage)), leverage, alpha=0.6, s=60, color='orange')
threshold_leverage = 2 * p_parametros / len(df)
axes[1, 0].axhline(y=threshold_leverage, color='r', linestyle='--', 
                   linewidth=2, label=f'Umbral: 2p/n = {threshold_leverage:.3f}')
axes[1, 0].set_xlabel('Observación ID', fontsize=10, fontweight='bold')
axes[1, 0].set_ylabel('Leverage (diagonal de matriz hat)', fontsize=10, fontweight='bold')
axes[1, 0].set_title('(C) Leverage: Extremidad en Espacio X', fontsize=11, fontweight='bold')
axes[1, 0].legend(fontsize=9)
axes[1, 0].grid(True, alpha=0.3)

# Panel 4: DFBETAS (cambio en coeficientes)
for i, col in enumerate(X_numeric.columns[:2]):  # Graficar primeros 2 predictores
    axes[1, 1].scatter(range(len(dfbetas)), dfbetas[:, i+1], alpha=0.6, s=50, label=col)
threshold_dfbetas = 2/np.sqrt(len(df))
axes[1, 1].axhline(y=threshold_dfbetas, color='r', linestyle='--', linewidth=1.5, label='Umbral')
axes[1, 1].axhline(y=-threshold_dfbetas, color='r', linestyle='--', linewidth=1.5)
axes[1, 1].set_xlabel('Observación ID', fontsize=10, fontweight='bold')
axes[1, 1].set_ylabel('DFBETAS', fontsize=10, fontweight='bold')
axes[1, 1].set_title('(D) DFBETAS: Influencia sobre Coeficientes', fontsize=11, fontweight='bold')
axes[1, 1].legend(fontsize=8, loc='best')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Identificar casos problemáticos
outlier_cooks = np.where(cooks_d > 4/len(df))[0]
outlier_residuals = np.where(np.abs(standardized_residuals) > 2)[0]
outlier_leverage = np.where(leverage > threshold_leverage)[0]

print(f"\n✓ Resumen de Observaciones Problemáticas:")
print(f"\n  Outliers (|resid studentizado| > 2): {len(outlier_residuals)} observaciones")
if len(outlier_residuals) > 0:
    print(f"    IDs base 0: {sorted(outlier_residuals[:10].tolist())} {'...' if len(outlier_residuals) > 10 else ''}")
    print(f"    id_pescador: {df.loc[outlier_residuals[:10], 'id_pescador'].astype(int).tolist()} {'...' if len(outlier_residuals) > 10 else ''}")
print(f"\n  Leverage alto (hat > {threshold_leverage:.3f}): {len(outlier_leverage)} observaciones")
if len(outlier_leverage) > 0:
    print(f"    IDs base 0: {sorted(outlier_leverage[:10].tolist())} {'...' if len(outlier_leverage) > 10 else ''}")
    print(f"    id_pescador: {df.loc[outlier_leverage[:10], 'id_pescador'].astype(int).tolist()} {'...' if len(outlier_leverage) > 10 else ''}")
print(f"\n  Cook's D alto (> {4/len(df):.3f}): {len(outlier_cooks)} observaciones")
if len(outlier_cooks) > 0:
    print(f"    IDs base 0: {sorted(outlier_cooks[:10].tolist())} {'...' if len(outlier_cooks) > 10 else ''}")
    print(f"    id_pescador: {df.loc[outlier_cooks[:10], 'id_pescador'].astype(int).tolist()} {'...' if len(outlier_cooks) > 10 else ''}")

print(f"\n✓ Acciones Recomendadas:")
print(f"  1. Verificar si outliers son errores de medición u observaciones válidas")
print(f"  2. Analizar sensibilidad: reajustar modelo sin observaciones influyentes")
print(f"  3. Si válidos: usar Robust Regression o Huber M-estimators")
print(f"  4. Documentar decisión en reporte")


## PARTE 9: RESUMEN COMPARATIVO DE MODELOS

In [ ]:
print("\n[PARTE 9] Comparación Final de Modelos Candidatos")
print("-" * 80)

# Compilar información de modelos
modelos = {
    'Simple (horas)': modelo_simple,
    'Múltiple (3 var)': modelo_multiple,
    'log(Y) + 3 var': modelo_log,
    'Escalado': modelo_scaled
}

# Tabla comparativa
comparison = []
for name, mod in modelos.items():
    comparison.append({
        'Modelo': name,
        'R²': f"{mod.rsquared:.4f}",
        'R² Adj': f"{mod.rsquared_adj:.4f}",
        'AIC': f"{mod.aic:.1f}",
        'BIC': f"{mod.bic:.1f}",
        'F-stat': f"{mod.fvalue:.2f}",
        'p-value': f"{mod.f_pvalue:.2e}"
    })

comparison_df = pd.DataFrame(comparison)
print("\n")
print(comparison_df.to_string(index=False))

print("\n✓ Criterios de Selección de Modelo:")
print("  • R²: Mayor es mejor, pero cuidado con sobreparametrización")
print("  • R² Ajustado: Penaliza modelos con muchos predictores")
print("  • AIC/BIC: Menor es mejor (penalizan complejidad)")
print("  • F-statistic: Significancia global del modelo")

print("\n✓ RECOMENDACIÓN: Modelo Múltiple (3 var)")
print("  Razones:")
print(f"    • R² = {modelo_multiple.rsquared:.4f} (mejora notable vs. simple)")
print(f"    • R² Adj = {modelo_multiple.rsquared_adj:.4f} (balance complejidad-ajuste)")
print(f"    • Interpretación clara de factores (horas, embarcación, experiencia)")
print(f"    • Supuestos verificados (aunque normalidad podría mejorarse con log)")
print(f"    • AIC = {modelo_multiple.aic:.1f} es competitivo")

## PARTE 10: CONSULTA AL TUTOR IA (EJEMPLOS DE PROMPTS)

### Ejemplos de Preguntas al Tutor IA para Situaciones Comunes

#### Ejemplo 1: Normalidad Violada

**Prompt:**
```
Mi modelo de regresión múltiple muestra residuales que NO pasan el test 
Shapiro-Wilk (p < 0.05). Los residuales están sesgados a la derecha.

¿Qué opciones tengo?
1. ¿Debo transformar Y? ¿Cuál transformación probaste?
2. ¿Puedo usar GLM en su lugar?
3. ¿Bootstrapping resuelve el problema?
4. ¿Hay métodos robustos que no asuman normalidad?

Mi Y es captura de peces (valores positivos, rango 5-150 kg).
```

**Respuesta esperada del Tutor:**
- Sugerir Box-Cox para encontrar transformación óptima
- Recomendar log(Y) para datos sesgados a la derecha
- Explicar GLM (Poisson o Gamma) como alternativa
- Bootstrapping para IC sin asumir normalidad
- Robust regression (Huber, M-estimators)

#### Ejemplo 2: Multicolinealidad Detectada

**Prompt:**
```
Mi matriz de correlaciones muestra que dos predictores tienen r = 0.87.
El VIF para ambas variables es > 10. ¿Cuál elimino o cómo las combino?

Variables: horas_faena (corr 0.87 con experiencia_anios)

¿Debería:
1. Eliminar una de las dos?
2. Crear un índice combinado?
3. Usar PCA o Ridge regression?
4. Dejar ambas si son teóricamente importantes?
```

**Respuesta esperada:**
- Verificar importancia teórica vs. estadística
- PCA para reducir dimensionalidad
- Ridge/Lasso regression
- Elimination stepwise
- Elasticnet para balance

#### Ejemplo 3: Outliers Influyentes

**Prompt:**
```
Encontré 3 observaciones con Cook's D > umbral (0.027).
Una de ellas tiene captura = 450 kg (outlier extremo).

¿Las elimino? ¿Cómo sé si son errores de medición u observaciones válidas?

¿Debo presentar resultados:
1. Con todas las observaciones?
2. Sin outliers?
3. Ambos y comparar?
```

**Respuesta esperada:**
- Sensitivity analysis: con/sin outliers
- Verificar dominio (¿450 kg es realista?)
- Robust regression si son válidos
- Documentar decisión
- Reportar ambos escenarios

## PARTE 11: TEMPLATE PARA TU ANÁLISIS

### 📋 CHECKLIST PARA PREPARAR TU LABORATORIO

**Antes de ejecutar este laboratorio con TUS datos, comparte:**

1️⃣ **Tu pregunta de investigación (tesis)**
   - Ejemplo: "¿Cómo afectan los factores socio-económicos (experiencia, educación) y técnicos (tamaño embarcación) a la captura de peces artesanales?"

2️⃣ **Variable Dependiente (Y)**
   - Nombre:
   - Tipo: (continua, conteo, proporción)
   - Rango/distribución esperada:
   - Unidades:

3️⃣ **Variables Independientes (X) Principales**
   - X₁: _____ (tipo: numérica/categórica, justificación teórica)
   - X₂: _____ 
   - X₃: _____ 
   - X₄ (opcional):

4️⃣ **Estructura de Datos**
   - N observaciones:
   - Fuente (primaria/secundaria, síntesis):
   - Ausentes (missing):

Luego te envío un **CHECKLIST PERSONALIZADO** con:
- Verificaciones previas (limpieza de datos)
- Gráficos exploratorios recomendados
- Transformaciones sugeridas
- Diagnósticos específicos para tu Y
- Tabla de resultados (formato APA)

In [ ]:
print("\n" + "="*80)
print("RESUMEN FINAL Y PRÓXIMOS PASOS")
print("="*80)
print(f"""
✅ LABORATORIO COMPLETADO

📊 Análisis realizados:
   ✓ Regresión lineal simple y múltiple
   ✓ Verificación de 4 supuestos del modelo lineal
   ✓ Diagnósticos visuales y tests estadísticos
   ✓ Análisis de multicolinealidad (VIF)
   ✓ Transformaciones de variables
   ✓ Detección de outliers e influencia
   ✓ Comparación de modelos candidatos

📁 Objetos generados en memoria:
   - df: Dataset completo
   - modelo_simple: Regresión 1 predictor
   - modelo_multiple: Regresión 3 predictores (RECOMENDADO)
   - modelo_log: Con transformación log(Y)
   - modelo_scaled: Con variables estandarizadas

🎯 PRÓXIMOS PASOS para TU ANÁLISIS:

1. Descarga este notebook y adapta el código a tu dataset
2. Reemplaza el dataset sintético con tus datos reales
3. Sigue el flujo: Exploración → Ajuste → Diagnósticos → Interpretación
4. Guarda todos los gráficos (resolución ≥ 300 dpi para entrega)
5. Documenta supuestos: ✓ OK / ⚠ Condicionado / ✗ Violado
6. Prepara tabla de coeficientes en formato APA

💡 TIPS:
   • Si normalidad falla: Transforma Y (log, sqrt, Box-Cox)
   • Si multicolinealidad: Usa VIF > 5 como alarma temprana
   • Si outliers: Reporta con/sin para sensitivity analysis
   • Si heterocedásticidad: Considera WLS o Robust SE

📞 Para consultas:
   - Usa el Tutor IA con prompts específicos
   - Comparte tu pregunta de tesis + variables
   - Te armaré un checklist personalizado

""")
print("="*80)
print(f"Fecha: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')}")
print("="*80)